In [7]:
import pandas as pd

df = pd.read_csv("../data/processed/temporal_account_features.csv")

print(df.shape)
print(df.columns.tolist())
print(df["churn_target"].value_counts())

(420, 19)
['account_id', 'account_name', 'industry', 'country', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'tenure_days', 'current_mrr', 'current_arr', 'active_seats', 'total_usage_count', 'ticket_count', 'avg_satisfaction', 'escalation_rate', 'has_upgraded', 'has_downgraded', 'churn_target']
churn_target
0    296
1    124
Name: count, dtype: int64


In [8]:
features = [
    "industry",
    "country",
    "referral_source",
    "plan_tier",
    "seats",
    "is_trial",
    "tenure_days",
    "current_mrr",
    "current_arr",
    "active_seats",
    "total_usage_count",
    "ticket_count",
    "avg_satisfaction",
    "escalation_rate",
    "has_upgraded",
    "has_downgraded"
]

X = df[features]
y = df["churn_target"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (420, 16)
y shape: (420,)

Target distribution:
churn_target
0    296
1    124
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())

Training set: (336, 16)
Test set: (84, 16)

Training target:
churn_target
0    237
1     99
Name: count, dtype: int64

Test target:
churn_target
0    59
1    25
Name: count, dtype: int64


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

categorical_features = [
    "industry",
    "country",
    "referral_source",
    "plan_tier"
]

numeric_features = [
    "seats",
    "is_trial",
    "tenure_days",
    "current_mrr",
    "current_arr",
    "active_seats",
    "total_usage_count",
    "ticket_count",
    "avg_satisfaction",
    "escalation_rate",
    "has_upgraded",
    "has_downgraded"
]

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", categorical_transformer, categorical_features),
    ("num", numeric_transformer, numeric_features)
])

In [11]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Logistic Regression training completed!")

Logistic Regression training completed!


In [12]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Predictions:", y_pred[:10])
print("Probabilities:", y_prob[:10])

Predictions: [1 0 0 0 0 0 0 0 0 1]
Probabilities: [0.71710197 0.36270485 0.42695061 0.28934081 0.42235118 0.2657643
 0.25029497 0.16615203 0.3901289  0.53518221]


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7023809523809523
Precision: 0.5
Recall   : 0.16
F1 Score : 0.24242424242424243
ROC-AUC  : 0.6122033898305085

Confusion Matrix:
[[55  4]
 [21  4]]


In [14]:
from xgboost import XGBClassifier

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        eval_metric="logloss"
    ))
])

In [15]:
xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")

XGBoost training completed!


In [16]:
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions:", xgb_pred[:10])
print("Probabilities:", xgb_prob[:10])

Predictions: [0 1 1 1 0 0 1 0 1 1]
Probabilities: [0.39872828 0.53832155 0.5476399  0.7063907  0.24167381 0.04701016
 0.53363204 0.09928335 0.528232   0.9197284 ]


In [17]:
print("Accuracy :", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred))
print("Recall   :", recall_score(y_test, xgb_pred))
print("F1 Score :", f1_score(y_test, xgb_pred))
print("ROC-AUC  :", roc_auc_score(y_test, xgb_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

Accuracy : 0.6547619047619048
Precision: 0.375
Recall   : 0.24
F1 Score : 0.2926829268292683
ROC-AUC  : 0.49559322033898306

Confusion Matrix:
[[49 10]
 [19  6]]


In [18]:
results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    },
    {
        "Model": "XGBoost",
        "Accuracy": accuracy_score(y_test, xgb_pred),
        "Precision": precision_score(y_test, xgb_pred),
        "Recall": recall_score(y_test, xgb_pred),
        "F1": f1_score(y_test, xgb_pred),
        "ROC-AUC": roc_auc_score(y_test, xgb_prob)
    }
])

print(results.round(4))

                 Model  Accuracy  Precision  Recall      F1  ROC-AUC
0  Logistic Regression    0.7024      0.500    0.16  0.2424   0.6122
1              XGBoost    0.6548      0.375    0.24  0.2927   0.4956


In [19]:
print(results.round(4).to_string(index=False))

              Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression    0.7024      0.500    0.16 0.2424   0.6122
            XGBoost    0.6548      0.375    0.24 0.2927   0.4956


In [20]:
print(results.round(4).to_string(index=False))

              Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression    0.7024      0.500    0.16 0.2424   0.6122
            XGBoost    0.6548      0.375    0.24 0.2927   0.4956


In [21]:
risk_scores = df.loc[X_test.index, [
    "account_id",
    "current_mrr",
    "current_arr"
]].copy()

risk_scores["churn_probability"] = y_prob

print(risk_scores.head(10))

    account_id  current_mrr  current_arr  churn_probability
358   A-58b9ff      20379.0     244548.0           0.717102
407   A-5f7781       4796.0      57552.0           0.362705
216   A-dbc825      26595.0     319140.0           0.426951
383   A-e7beab       4982.0      59784.0           0.289341
194   A-9bfc9f      12745.0     152940.0           0.422351
187   A-cab532      14243.0     170916.0           0.265764
333   A-940b8b       9640.0     115680.0           0.250295
96    A-92a333      10659.0     127908.0           0.166152
368   A-716841       7469.0      89628.0           0.390129
198   A-ab438f          0.0          0.0           0.535182


In [22]:
risk_scores["revenue_at_risk"] = (
    risk_scores["churn_probability"] * risk_scores["current_mrr"]
)

print(
    risk_scores.sort_values(
        "revenue_at_risk",
        ascending=False
    ).head(10)
)

    account_id  current_mrr  current_arr  churn_probability  revenue_at_risk
358   A-58b9ff      20379.0     244548.0           0.717102     14613.821098
45    A-4c38bc      29520.0     354240.0           0.390992     11542.082886
216   A-dbc825      26595.0     319140.0           0.426951     11354.751430
381   A-054577      23577.0     282924.0           0.456232     10756.588407
63    A-02cd81      28644.0     343728.0           0.327949      9393.771972
100   A-65aeb5      32430.0     389160.0           0.261241      8472.045962
69    A-726cfa      20785.0     249420.0           0.401582      8346.879381
67    A-98a59a      33712.0     404544.0           0.244926      8256.959245
235   A-4e960a      23126.0     277512.0           0.353208      8168.278316
179   A-f03140      15702.0     188424.0           0.504838      7926.961790


In [23]:
risk_scores["priority"] = "Low"

risk_scores.loc[
    (risk_scores["churn_probability"] >= 0.50) |
    (risk_scores["revenue_at_risk"] >= 7500),
    "priority"
] = "High"

risk_scores.loc[
    (
        (risk_scores["churn_probability"] >= 0.30) &
        (risk_scores["churn_probability"] < 0.50)
    ) &
    (risk_scores["revenue_at_risk"] < 7500),
    "priority"
] = "Medium"

print(
    risk_scores[
        ["account_id", "churn_probability",
         "revenue_at_risk", "priority"]
    ]
    .sort_values("revenue_at_risk", ascending=False)
    .head(15)
)

    account_id  churn_probability  revenue_at_risk priority
358   A-58b9ff           0.717102     14613.821098     High
45    A-4c38bc           0.390992     11542.082886     High
216   A-dbc825           0.426951     11354.751430     High
381   A-054577           0.456232     10756.588407     High
63    A-02cd81           0.327949      9393.771972     High
100   A-65aeb5           0.261241      8472.045962     High
69    A-726cfa           0.401582      8346.879381     High
67    A-98a59a           0.244926      8256.959245     High
235   A-4e960a           0.353208      8168.278316     High
179   A-f03140           0.504838      7926.961790     High
263   A-883b7d           0.263606      7516.716690     High
240   A-5a3eb9           0.469751      7248.264593   Medium
171   A-3be56b           0.472462      7060.946743   Medium
395   A-d77f4c           0.157914      6704.231286      Low
350   A-659280           0.312707      6606.561741   Medium


In [24]:
risk_scores = risk_scores.merge(
    df[
        [
            "account_id",
            "avg_satisfaction",
            "ticket_count",
            "escalation_rate",
            "total_usage_count",
            "has_upgraded",
            "has_downgraded",
            "plan_tier"
        ]
    ],
    on="account_id",
    how="left"
)

def recommend_action(row):
    if row["escalation_rate"] >= 0.3:
        return "Priority support intervention"
    elif pd.notna(row["avg_satisfaction"]) and row["avg_satisfaction"] <= 3:
        return "Customer satisfaction outreach"
    elif row["has_downgraded"]:
        return "Account retention review"
    elif row["total_usage_count"] < 200:
        return "Product adoption outreach"
    else:
        return "Proactive retention outreach"

risk_scores["recommended_action"] = risk_scores.apply(
    recommend_action,
    axis=1
)

print(
    risk_scores[
        [
            "account_id",
            "churn_probability",
            "revenue_at_risk",
            "priority",
            "recommended_action"
        ]
    ]
    .sort_values("revenue_at_risk", ascending=False)
    .head(15)
)

   account_id  churn_probability  revenue_at_risk priority  \
0    A-58b9ff           0.717102     14613.821098     High   
38   A-4c38bc           0.390992     11542.082886     High   
2    A-dbc825           0.426951     11354.751430     High   
73   A-054577           0.456232     10756.588407     High   
70   A-02cd81           0.327949      9393.771972     High   
41   A-65aeb5           0.261241      8472.045962     High   
20   A-726cfa           0.401582      8346.879381     High   
34   A-98a59a           0.244926      8256.959245     High   
26   A-4e960a           0.353208      8168.278316     High   
65   A-f03140           0.504838      7926.961790     High   
28   A-883b7d           0.263606      7516.716690     High   
32   A-5a3eb9           0.469751      7248.264593   Medium   
62   A-3be56b           0.472462      7060.946743   Medium   
12   A-d77f4c           0.157914      6704.231286      Low   
36   A-659280           0.312707      6606.561741   Medium   

       

In [25]:
risk_scores.to_csv(
    "../data/processed/customer_risk_decisions.csv",
    index=False
)

print("Saved customer risk and decision dataset!")
print("Rows:", len(risk_scores))

Saved customer risk and decision dataset!
Rows: 84


In [26]:
risk_scores.to_csv(
    "../data/processed/customer_risk_decisions.csv",
    index=False
)

print("Saved customer risk and decision dataset!")
print("Rows:", len(risk_scores))

Saved customer risk and decision dataset!
Rows: 84


In [27]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/churn_risk_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [28]:
def explain_customer(account_id):
    customer = risk_scores[
        risk_scores["account_id"] == account_id
    ]

    if customer.empty:
        return "Customer not found."

    row = customer.iloc[0]

    print("Customer:", row["account_id"])
    print(f"Churn probability: {row['churn_probability']:.2%}")
    print(f"Revenue at risk: ${row['revenue_at_risk']:,.2f}")
    print("Priority:", row["priority"])
    print("Recommended action:", row["recommended_action"])

    print("\nRisk signals:")

    if row["churn_probability"] >= 0.5:
        print("- High predicted churn probability")

    if row["revenue_at_risk"] >= 7500:
        print("- Significant revenue exposure")

    if row["escalation_rate"] >= 0.3:
        print("- High support escalation rate")

    if pd.notna(row["avg_satisfaction"]) and row["avg_satisfaction"] <= 3:
        print("- Low customer satisfaction")

    if row["has_downgraded"]:
        print("- Customer has downgraded")

    if row["total_usage_count"] < 200:
        print("- Low product usage")

In [29]:
explain_customer("A-58b9ff")

Customer: A-58b9ff
Churn probability: 71.71%
Revenue at risk: $14,613.82
Priority: High
Recommended action: Priority support intervention

Risk signals:
- High predicted churn probability
- Significant revenue exposure
- High support escalation rate
- Customer has downgraded


In [30]:
def get_customer_context(account_id):
    customer = risk_scores[
        risk_scores["account_id"] == account_id
    ]

    if customer.empty:
        return None

    row = customer.iloc[0]

    return {
        "account_id": row["account_id"],
        "churn_probability": row["churn_probability"],
        "revenue_at_risk": row["revenue_at_risk"],
        "priority": row["priority"],
        "recommended_action": row["recommended_action"],
        "avg_satisfaction": row["avg_satisfaction"],
        "ticket_count": row["ticket_count"],
        "escalation_rate": row["escalation_rate"],
        "total_usage_count": row["total_usage_count"],
        "has_upgraded": row["has_upgraded"],
        "has_downgraded": row["has_downgraded"],
        "plan_tier": row["plan_tier"]
    }

In [31]:
context = get_customer_context("A-58b9ff")

print(context)

{'account_id': 'A-58b9ff', 'churn_probability': np.float64(0.7171019725375218), 'revenue_at_risk': np.float64(14613.821098342158), 'priority': 'High', 'recommended_action': 'Priority support intervention', 'avg_satisfaction': np.float64(4.0), 'ticket_count': np.float64(2.0), 'escalation_rate': np.float64(0.5), 'total_usage_count': np.int64(581), 'has_upgraded': np.False_, 'has_downgraded': np.True_, 'plan_tier': 'Enterprise'}


In [32]:
def customer_summary(account_id):
    context = get_customer_context(account_id)

    if context is None:
        return "Customer not found."

    return f"""
Customer {context['account_id']} has a predicted churn probability of
{context['churn_probability']:.1%}.

The estimated monthly revenue at risk is
${context['revenue_at_risk']:,.2f}.

Priority level: {context['priority']}

Recommended action: {context['recommended_action']}.

Supporting signals:
- Support tickets: {context['ticket_count']}
- Escalation rate: {context['escalation_rate']:.1%}
- Average satisfaction: {context['avg_satisfaction']}
- Total usage: {context['total_usage_count']}
- Upgraded previously: {context['has_upgraded']}
- Downgraded previously: {context['has_downgraded']}
- Plan: {context['plan_tier']}
"""

In [33]:
print(customer_summary("A-58b9ff"))


Customer A-58b9ff has a predicted churn probability of
71.7%.

The estimated monthly revenue at risk is
$14,613.82.

Priority level: High

Recommended action: Priority support intervention.

Supporting signals:
- Support tickets: 2.0
- Escalation rate: 50.0%
- Average satisfaction: 4.0
- Total usage: 581
- Upgraded previously: False
- Downgraded previously: True
- Plan: Enterprise



In [34]:
def ask_assistant(account_id, question):
    context = get_customer_context(account_id)

    if context is None:
        return "Customer not found."

    question = question.lower()

    if "why" in question or "risk" in question:
        return customer_summary(account_id)

    elif "revenue" in question or "money" in question:
        return (
            f"Customer {account_id} has an estimated "
            f"${context['revenue_at_risk']:,.2f} in monthly revenue at risk."
        )

    elif "action" in question or "do" in question:
        return (
            f"The recommended action for {account_id} is: "
            f"{context['recommended_action']}."
        )

    else:
        return customer_summary(account_id)

In [35]:
print(
    ask_assistant(
        "A-58b9ff",
        "Why is this customer high risk?"
    )
)


Customer A-58b9ff has a predicted churn probability of
71.7%.

The estimated monthly revenue at risk is
$14,613.82.

Priority level: High

Recommended action: Priority support intervention.

Supporting signals:
- Support tickets: 2.0
- Escalation rate: 50.0%
- Average satisfaction: 4.0
- Total usage: 581
- Upgraded previously: False
- Downgraded previously: True
- Plan: Enterprise



In [36]:
def top_priority_customers(n=10):
    return (
        risk_scores[
            [
                "account_id",
                "churn_probability",
                "revenue_at_risk",
                "priority",
                "recommended_action"
            ]
        ]
        .sort_values("revenue_at_risk", ascending=False)
        .head(n)
    )

In [37]:
print(top_priority_customers(10))

   account_id  churn_probability  revenue_at_risk priority  \
0    A-58b9ff           0.717102     14613.821098     High   
38   A-4c38bc           0.390992     11542.082886     High   
2    A-dbc825           0.426951     11354.751430     High   
73   A-054577           0.456232     10756.588407     High   
70   A-02cd81           0.327949      9393.771972     High   
41   A-65aeb5           0.261241      8472.045962     High   
20   A-726cfa           0.401582      8346.879381     High   
34   A-98a59a           0.244926      8256.959245     High   
26   A-4e960a           0.353208      8168.278316     High   
65   A-f03140           0.504838      7926.961790     High   

                recommended_action  
0    Priority support intervention  
38    Proactive retention outreach  
2   Customer satisfaction outreach  
73        Account retention review  
70   Priority support intervention  
41        Account retention review  
20    Proactive retention outreach  
34        Account ret

In [38]:
def ask_assistant(question, account_id=None):
    question_lower = question.lower()

    # Portfolio-level question
    if account_id is None and (
        "prioritize" in question_lower
        or "highest risk" in question_lower
        or "top customers" in question_lower
    ):
        top = top_priority_customers(10)

        return top.to_string(index=False)

    # Customer-level question
    if account_id is not None:
        return ask_assistant_customer(account_id, question)

    return "Please provide a customer ID or ask a portfolio-level question."

In [39]:
ask_assistant_customer = ask_assistant

In [40]:
def customer_assistant(account_id, question):
    context = get_customer_context(account_id)

    if context is None:
        return "Customer not found."

    question = question.lower()

    if "revenue" in question or "money" in question:
        return (
            f"Customer {account_id} has an estimated "
            f"${context['revenue_at_risk']:,.2f} in monthly revenue at risk."
        )

    elif "action" in question or "do" in question:
        return (
            f"Recommended action for {account_id}: "
            f"{context['recommended_action']}."
        )

    elif "why" in question or "risk" in question:
        return customer_summary(account_id)

    else:
        return customer_summary(account_id)


def decision_assistant(question, account_id=None):
    question_lower = question.lower()

    # Portfolio-level question
    if account_id is None:
        if (
            "prioritize" in question_lower
            or "highest risk" in question_lower
            or "top customers" in question_lower
        ):
            return top_priority_customers(10)

        return "Please provide a customer ID for a customer-specific question."

    # Customer-level question
    return customer_assistant(account_id, question)

In [41]:
print(
    decision_assistant(
        "Why is this customer high risk?",
        "A-58b9ff"
    )
)


Customer A-58b9ff has a predicted churn probability of
71.7%.

The estimated monthly revenue at risk is
$14,613.82.

Priority level: High

Recommended action: Priority support intervention.

Supporting signals:
- Support tickets: 2.0
- Escalation rate: 50.0%
- Average satisfaction: 4.0
- Total usage: 581
- Upgraded previously: False
- Downgraded previously: True
- Plan: Enterprise



In [42]:
print(
    decision_assistant(
        "Which customers should we prioritize?"
    )
)

   account_id  churn_probability  revenue_at_risk priority  \
0    A-58b9ff           0.717102     14613.821098     High   
38   A-4c38bc           0.390992     11542.082886     High   
2    A-dbc825           0.426951     11354.751430     High   
73   A-054577           0.456232     10756.588407     High   
70   A-02cd81           0.327949      9393.771972     High   
41   A-65aeb5           0.261241      8472.045962     High   
20   A-726cfa           0.401582      8346.879381     High   
34   A-98a59a           0.244926      8256.959245     High   
26   A-4e960a           0.353208      8168.278316     High   
65   A-f03140           0.504838      7926.961790     High   

                recommended_action  
0    Priority support intervention  
38    Proactive retention outreach  
2   Customer satisfaction outreach  
73        Account retention review  
70   Priority support intervention  
41        Account retention review  
20    Proactive retention outreach  
34        Account ret

In [43]:
def build_ai_prompt(account_id, question):
    context = get_customer_context(account_id)

    if context is None:
        return None

    prompt = f"""
You are a Revenue Recovery and Customer Risk Assistant.

Answer the user's question using ONLY the customer data provided below.
Do not invent facts or numbers.

Customer data:
- Account ID: {context['account_id']}
- Plan: {context['plan_tier']}
- Churn probability: {context['churn_probability']:.2%}
- Revenue at risk: ${context['revenue_at_risk']:,.2f}
- Priority: {context['priority']}
- Recommended action: {context['recommended_action']}
- Support tickets: {context['ticket_count']}
- Escalation rate: {context['escalation_rate']:.2%}
- Average satisfaction: {context['avg_satisfaction']}
- Total usage: {context['total_usage_count']}
- Previously upgraded: {context['has_upgraded']}
- Previously downgraded: {context['has_downgraded']}

User question:
{question}

Give a concise, business-focused answer.
Clearly distinguish model predictions from observed customer information.
"""

    return prompt

In [44]:
prompt = build_ai_prompt(
    "A-58b9ff",
    "Why is this customer high risk?"
)

print(prompt)


You are a Revenue Recovery and Customer Risk Assistant.

Answer the user's question using ONLY the customer data provided below.
Do not invent facts or numbers.

Customer data:
- Account ID: A-58b9ff
- Plan: Enterprise
- Churn probability: 71.71%
- Revenue at risk: $14,613.82
- Priority: High
- Recommended action: Priority support intervention
- Support tickets: 2.0
- Escalation rate: 50.00%
- Average satisfaction: 4.0
- Total usage: 581
- Previously upgraded: False
- Previously downgraded: True

User question:
Why is this customer high risk?

Give a concise, business-focused answer.
Clearly distinguish model predictions from observed customer information.



In [45]:
import os

print(os.getenv("OPENAI_API_KEY") is not None)

True


In [46]:
$env:OPENAI_API_KEY="REDACTED_API_KEY"

SyntaxError: invalid syntax (3162344999.py, line 1)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("OPENAI_API_KEY") is not None)

True


In [ ]:
from openai import OpenAI

client = OpenAI()

print("OpenAI client initialized!")

OpenAI client initialized!


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.6-luna",
    input="Say hello in one sentence."
)

print(response.output_text)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_NEW****_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print(os.getenv("OPENAI_API_KEY") is not None)

True


In [ ]:
from openai import OpenAI

client = OpenAI()

print("OpenAI client initialized!")

OpenAI client initialized!


In [ ]:
def ask_ai_assistant(account_id, question):
    prompt = build_ai_prompt(account_id, question)

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    return response.output_text

In [ ]:
print(
    ask_ai_assistant(
        "A-58b9ff",
        "Why is this customer high risk?"
    )
)

NameError: name 'build_ai_prompt' is not defined

In [ ]:
def build_ai_prompt(account_id, question):
    context = get_customer_context(account_id)

    if context is None:
        return f"Customer {account_id} was not found."

    return f"""
You are a Revenue Recovery and Customer Risk Assistant.

Answer the user's question using ONLY the customer data provided below.
Do not invent facts or numbers.

Customer data:
- Account ID: {context['account_id']}
- Plan: {context['plan_tier']}
- Churn probability: {context['churn_probability']:.2%}
- Revenue at risk: ${context['revenue_at_risk']:,.2f}
- Priority: {context['priority']}
- Recommended action: {context['recommended_action']}
- Support tickets: {context['ticket_count']}
- Escalation rate: {context['escalation_rate']:.2%}
- Average satisfaction: {context['avg_satisfaction']}
- Total usage: {context['total_usage_count']}
- Has upgraded: {context['has_upgraded']}
- Has downgraded: {context['has_downgraded']}

User question:
{question}

Give a concise, business-focused answer.
Clearly distinguish model predictions from observed customer information.
"""

In [47]:
print(build_ai_prompt(
    "A-58b9ff",
    "Why is this customer high risk?"
))


You are a Revenue Recovery and Customer Risk Assistant.

Answer the user's question using ONLY the customer data provided below.
Do not invent facts or numbers.

Customer data:
- Account ID: A-58b9ff
- Plan: Enterprise
- Churn probability: 71.71%
- Revenue at risk: $14,613.82
- Priority: High
- Recommended action: Priority support intervention
- Support tickets: 2.0
- Escalation rate: 50.00%
- Average satisfaction: 4.0
- Total usage: 581
- Previously upgraded: False
- Previously downgraded: True

User question:
Why is this customer high risk?

Give a concise, business-focused answer.
Clearly distinguish model predictions from observed customer information.



In [48]:
def ask_ai_assistant(account_id, question):
    prompt = build_ai_prompt(account_id, question)

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    return response.output_text

In [49]:
print(ask_ai_assistant(
    "A-58b9ff",
    "Why is this customer high risk?"
))

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: $env:OPE****************************************************************************************************************************************************************************SQ8A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [50]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

key = os.getenv("OPENAI_API_KEY")

print("Key loaded:", key is not None)
print("Starts correctly:", key.startswith("sk-") if key else False)

Key loaded: True
Starts correctly: False


In [51]:
key = os.getenv("OPENAI_API_KEY")

print("Length:", len(key) if key else 0)
print("First 4 characters:", key[:4] if key else "None")

Length: 184
First 4 characters: $env


In [53]:
from dotenv import load_dotenv
load_dotenv(override=True)

key = os.getenv("OPENAI_API_KEY")
print(key[:4] if key else "None")

$env


In [54]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

key = os.getenv("OPENAI_API_KEY")

print("First 4:", key[:4] if key else "None")
print("Length:", len(key) if key else 0)

First 4: REDACTED_API_KEY
Length: 164


In [55]:
from openai import OpenAI

client = OpenAI()

print("Client ready!")

Client ready!


In [56]:
def ask_ai_assistant(account_id, question):
    prompt = build_ai_prompt(account_id, question)

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    return response.output_text

In [57]:
print(
    ask_ai_assistant(
        "A-58b9ff",
        "Why is this customer high risk?"
    )
)

This customer is high risk because the model predicts a **71.71% probability of churn**, putting **$14,613.82 in revenue at risk**.

Observed risk signals include:
- **Previously downgraded**
- **50% escalation rate** across 2 support tickets

Their **4.0 average satisfaction** and **581 total usage** are observed metrics but are not, by themselves, evidence of elevated risk without benchmarks.


In [58]:
print(
    ask_ai_assistant(
        "A-58b9ff",
        "How much revenue is at risk?"
    )
)

Model-predicted revenue at risk is **$14,613.82**.


In [59]:
print(
    ask_ai_assistant(
        "A-58b9ff",
        "What action should the company take?"
    )
)

The company should initiate a **priority support intervention**.

- **Model prediction:** The account has a **71.71% churn probability** and **$14,613.82 in revenue at risk**, warranting high priority.
- **Observed information:** The customer has **2 support tickets**, a **50% escalation rate**, and has previously downgraded, despite an average satisfaction score of **4.0**.


In [65]:
def build_business_prompt(question):
    top_customers = (
        customer_risk_decisions
        .sort_values("revenue_at_risk", ascending=False)
        .head(10)
    )

    customer_data = top_customers[
        [
            "account_id",
            "churn_probability",
            "revenue_at_risk",
            "priority",
            "recommended_action"
        ]
    ].to_string(index=False)

    return f"""
You are a Revenue Recovery Decision Intelligence Assistant.

Answer the business question using ONLY the customer data below.
Do not invent customers, numbers, or facts.

Top customers by estimated revenue at risk:

{customer_data}

Business question:
{question}

Give a concise business-focused answer.
Explain the reasoning using the available numbers.
"""

In [66]:
def ask_business_ai(question):
    prompt = build_business_prompt(question)

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    return response.output_text

In [67]:
print(
    ask_business_ai(
        "Which customers should we prioritize and why?"
    )
)

Prioritize customers by **revenue at risk**, using churn probability to identify urgency:

1. **A-58b9ff** — **$14,613.82 at risk**, **71.7% churn probability**. Highest-value and highest-risk account; initiate **priority support intervention** immediately.
2. **A-4c38bc** — **$11,542.08 at risk**, **39.1% probability**. Use **proactive retention outreach**.
3. **A-dbc825** — **$11,354.75 at risk**, **42.7% probability**. Conduct **customer satisfaction outreach**.
4. **A-054577** — **$10,756.59 at risk**, **45.6% probability**. Perform an **account retention review**.
5. **A-02cd81** — **$9,393.77 at risk**, **32.8% probability**. Provide **priority support intervention**.

Next priority group:

- **A-65aeb5** — $8,472.05 at risk; 26.1% probability
- **A-726cfa** — $8,346.88; 40.2%
- **A-98a59a** — $8,256.96; 24.5%
- **A-4e960a** — $8,168.28; 35.3%
- **A-f03140** — $7,926.96; 50.5%

These should receive **account retention reviews**, with A-f03140 elevated within this group because it

In [63]:
import pandas as pd

customer_risk_decisions = pd.read_csv(
    "../data/processed/customer_risk_decisions.csv"
)

print(customer_risk_decisions.shape)

(84, 14)


In [64]:
customer_risk_decisions.head()

,account_id,current_mrr,current_arr,churn_probability,revenue_at_risk,priority,avg_satisfaction,ticket_count,escalation_rate,total_usage_count,has_upgraded,has_downgraded,plan_tier,recommended_action
0,A-58b9ff,20379.0,244548.0,0.717102,14613.821098,High,4.000000,2.0,0.500000,581,False,True,Enterprise,Priority support intervention
1,A-5f7781,4796.0,57552.0,0.362705,1739.532460,Medium,4.000000,2.0,0.000000,169,True,False,Pro,Product adoption outreach
2,A-dbc825,26595.0,319140.0,0.426951,11354.751430,High,3.000000,5.0,0.000000,612,True,False,Enterprise,Customer satisfaction outreach
3,A-e7beab,4982.0,59784.0,0.289341,1441.495917,Low,4.000000,6.0,0.000000,356,False,False,Enterprise,Proactive retention outreach
4,A-9bfc9f,12745.0,152940.0,0.422351,5382.865781,Medium,3.333333,6.0,0.166667,357,True,True,Pro,Account retention review


In [68]:
import pandas as pd
import joblib

df = pd.read_csv("../data/processed/temporal_account_features.csv")

model = joblib.load("../models/churn_risk_model.joblib")

print("Customers:", len(df))
print("Model loaded:", type(model).__name__)

Customers: 420
Model loaded: Pipeline


In [69]:
feature_cols = [
    "industry",
    "country",
    "referral_source",
    "plan_tier",
    "seats",
    "is_trial",
    "tenure_days",
    "current_mrr",
    "current_arr",
    "active_seats",
    "total_usage_count",
    "ticket_count",
    "avg_satisfaction",
    "escalation_rate",
    "has_upgraded",
    "has_downgraded"
]

X_all = df[feature_cols]

df["churn_probability"] = model.predict_proba(X_all)[:, 1]

In [70]:
df["revenue_at_risk"] = (
    df["churn_probability"] * df["current_mrr"]
)

In [71]:
def assign_priority(row):
    if (
        row["churn_probability"] >= 0.50
        or row["revenue_at_risk"] >= 7500
    ):
        return "High"
    elif (
        0.30 <= row["churn_probability"] < 0.50
        and row["revenue_at_risk"] < 7500
    ):
        return "Medium"
    else:
        return "Low"

df["priority"] = df.apply(assign_priority, axis=1)

In [72]:
def recommend_action(row):
    if row["escalation_rate"] >= 0.3:
        return "Priority support intervention"
    elif pd.notna(row["avg_satisfaction"]) and row["avg_satisfaction"] <= 3:
        return "Customer satisfaction outreach"
    elif row["has_downgraded"]:
        return "Account retention review"
    elif row["total_usage_count"] < 200:
        return "Product adoption outreach"
    else:
        return "Proactive retention outreach"

df["recommended_action"] = df.apply(recommend_action, axis=1)

In [73]:
final_decisions = df[
    [
        "account_id",
        "account_name",
        "plan_tier",
        "current_mrr",
        "current_arr",
        "churn_probability",
        "revenue_at_risk",
        "priority",
        "recommended_action",
        "avg_satisfaction",
        "ticket_count",
        "escalation_rate",
        "total_usage_count",
        "has_upgraded",
        "has_downgraded"
    ]
]

final_decisions.to_csv(
    "../data/processed/customer_risk_decisions.csv",
    index=False
)

print("Saved:", final_decisions.shape)

Saved: (420, 15)
